# AI music detector — demo

Upload a song and see how it ranks among real songs, plus the **fakeprint** the decision is based on.

**How it works (short):** AI music generators build audio with upsampling layers, which leave tiny, regular peaks in
the average spectrum. The *fakeprint* keeps only those peaks (after Afchar et al., ISMIR 2025). A logistic regression
scores it, and the score is ranked among real songs the model never trained on.

**How to read the answer**
- *higher than X% of real songs*: how unusual the song is for a real song. Above 99% = **likely AI**, 90–99% = **unsure**, below = **likely real**.
- Files at 44.1 kHz or more: up to the first 4 minutes are cut into 30 s parts, each part is scored, and the middle score counts. (`demo.show(path, mode="whole")` scores the whole song instead: it catches more AI but flagged 7% of new real songs, not 5%.)
- Lower sample rates use a simpler 16 kHz model that only says AI / real.
- This is a research project, not proof. About 5% of real songs rank above 95%, and new generators are harder to catch.

## 1. Setup

In [ ]:
!pip -q install soundfile plotly

Upload **`ai-music-detector-demo.zip`** (the `dist` folder of the project) when asked. It holds the code and the
models as plain numbers (no pickles), so the Colab library versions don't matter.

In [ ]:
import os, zipfile
if not os.path.exists("ai-music-detector"):
    from google.colab import files
    uploaded = files.upload()          # choose ai-music-detector-demo.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")
# Or, if the project is on GitHub:  !git clone https://github.com/<user>/<repo> ai-music-detector
%cd ai-music-detector
from demo import show

## 2. Try it on two synthetic clips (no upload needed)

`synth.py` makes a "real" clip (instruments and noise) and an "AI" clip (the same kind of audio passed through a random
upsampling decoder, which leaves a comb of peaks). The AI decoder was never trained on, so it is a hard case.

In [ ]:
import numpy as np, soundfile as sf, synth
rng = np.random.default_rng(0)
for name, make in (("synth_real.wav", synth.make_real_clip), ("synth_ai.wav", synth.make_ai_clip)):
    x = make(rng, sr=44100, dur=65.0)
    sf.write(name, np.stack([x, x], 1) * 0.5 / np.abs(x).max(), 44100)
    show(name)

## 3. Score your own songs

Upload one or more files (MP3, WAV, FLAC, OGG). `base_rate`: the share of AI songs where you would use the detector
(0 = off); it adds a rough chance that the song is AI. `joint`: also use the precise comb (better for new
generators that leave a comb).

In [ ]:
base_rate = 0.1   #@param {type:"number"}
joint = False     #@param {type:"boolean"}
from google.colab import files
for name in files.upload():
    show(name, base_rate=base_rate, joint=joint)

## Notes

- The model is trained on 30 s clips from ~16k songs (real: FMA, netlabels; AI: Suno, Udio, MusicGen, Stable Audio,
  ElevenLabs and 8 more). Whole songs get their own cut-off, because a cleaner fakeprint raises every score.
- *Rough chance*: compares how often AI and real songs land in the same score range. It is too low for generators the
  model never saw.
- *peaks repeat every N Hz*: an exact comb was found; known spacings are named (e.g. 200 Hz: Suno / ElevenLabs).
- Speed changes of about 1% break the classifier (the comb survives). See RESEARCH_LOG.md in the project for all tests.